# Блок Прогноз — Unsupervised notebook

**Autoencoder + Gaussian Mixture + Isolation Forest**

Этот ноутбук собирает итоговый unsupervised-подход, который мы обсуждали в чате:

- без pseudo-labels и без supervised-моделей;
- признаки строятся только из геологических факторов;
- используются Autoencoder, GMM и Isolation Forest;
- итоговая оценка дополнительно фильтруется через `support_score`;
- карта красится в стиле, похожем на пример из ГИС Интегро: `prognoz`, где **0 = наиболее перспективно**.


In [ ]:

# =========================
# 1. Импорты
# =========================
import os
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from shapely.ops import unary_union

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:

# =========================
# 2. Пути
# =========================
BASE_DIR = Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз")
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "unsupervised_forecast_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("SHP_DIR :", SHP_DIR)
print("OUT_DIR :", OUT_DIR)


In [ ]:

# =========================
# 3. Настройки модели
# =========================
CELL_SIZE = 500  # 500 x 500 м

# Масштабы близости 1 / (1 + d/scale)
SCALE_FACIES = 1000.0
SCALE_PALEO  = 1000.0
SCALE_STRUCT = 900.0
SCALE_MAGM   = 800.0
SCALE_TECT1  = 800.0
SCALE_TECT2  = 800.0

# Autoencoder (через MLPRegressor: вход -> bottleneck -> выход)
AE_HIDDEN = (12, 4, 12)
AE_MAX_ITER = 1500

# GMM
GMM_COMPONENTS_RANGE = range(2, 8)

# Isolation Forest
IFOREST_CONTAMINATION = 0.08

# Итоговая формула
TOP_Q = 0.94          # верхние ~6% зон
YELLOW_LIMIT = 0.15   # в стиле ГИС Интегро: prognoz <= 0.15

# Граница маленькой геологической поддержки
SUPPORT_POWER = 0.65

print("Настройки загружены.")


In [ ]:

# =========================
# 4. Утилиты
# =========================
def normalize_01(x):
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    if mask.sum() == 0:
        return np.zeros_like(x, dtype=float)
    xmin = np.nanmin(x[mask])
    xmax = np.nanmax(x[mask])
    if not np.isfinite(xmin) or not np.isfinite(xmax) or abs(xmax - xmin) < 1e-12:
        out = np.zeros_like(x, dtype=float)
        out[mask] = 0.5
        return out
    out = (x - xmin) / (xmax - xmin)
    out[~mask] = 0
    return np.clip(out, 0, 1)

def safe_set_crs(gdf, fallback="EPSG:4326"):
    if gdf.crs is None:
        gdf = gdf.set_crs(fallback)
    return gdf

def repair_geometries(gdf):
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna()].copy()
    if len(gdf) == 0:
        return gdf
    try:
        gdf["geometry"] = gdf.buffer(0)
    except Exception:
        pass
    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
    return gdf

def load_layer(path, target_crs=None):
    gdf = gpd.read_file(path)
    gdf = safe_set_crs(gdf, "EPSG:4326")
    gdf = repair_geometries(gdf)
    if target_crs is not None and gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
    return gdf

def finite_bounds(gdf, layer_name="layer"):
    bounds = np.asarray(gdf.total_bounds, dtype=float)
    if len(bounds) != 4 or not np.all(np.isfinite(bounds)):
        raise ValueError(f"Некорректные total_bounds у {layer_name}: {bounds}")
    minx, miny, maxx, maxy = bounds
    if maxx <= minx or maxy <= miny:
        raise ValueError(f"Пустые границы у {layer_name}: {bounds}")
    return minx, miny, maxx, maxy

def build_grid(mask_gdf, cell_size=500):
    mask_gdf = repair_geometries(mask_gdf)
    minx, miny, maxx, maxy = finite_bounds(mask_gdf, "mask")
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    cells = []
    ids = []
    idx = 0
    for x in xs:
        for y in ys:
            cells.append(box(x, y, x + cell_size, y + cell_size))
            ids.append(idx)
            idx += 1

    grid = gpd.GeoDataFrame({"cell_id": ids}, geometry=cells, crs=mask_gdf.crs)

    mask_union = unary_union(mask_gdf.geometry)
    grid = grid[grid.intersects(mask_union)].copy()
    grid.reset_index(drop=True, inplace=True)

    grid["centroid"] = grid.geometry.centroid
    grid["cx"] = grid["centroid"].x
    grid["cy"] = grid["centroid"].y
    return grid

def union_geometry(gdf):
    gdf = repair_geometries(gdf)
    if len(gdf) == 0:
        return None
    return unary_union(gdf.geometry)

def distance_to_union(points_gdf, geom_union):
    if geom_union is None:
        return np.full(len(points_gdf), np.nan)
    return points_gdf.geometry.distance(geom_union).to_numpy()

def proximity_from_distance(dist, scale):
    dist = np.asarray(dist, dtype=float)
    out = 1.0 / (1.0 + dist / float(scale))
    out[~np.isfinite(out)] = 0.0
    return np.clip(out, 0, 1)

def try_find_layer(layer_files, include_names):
    include_names = [s.lower() for s in include_names]
    for stem, path in layer_files.items():
        s = stem.lower()
        if any(k in s for k in include_names):
            return path
    return None

def make_points_from_layer(gdf):
    if len(gdf) == 0:
        return gdf
    geom_types = set(gdf.geom_type.astype(str).str.lower())
    if geom_types <= {"point", "multipoint"}:
        pts = gdf.explode(index_parts=False).copy()
        pts = pts[pts.geometry.type.isin(["Point"])].copy()
        return pts
    out = gdf.copy()
    out["geometry"] = out.representative_point()
    return out

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


In [ ]:

# =========================
# 5. Поиск слоев
# =========================
layer_files = {}
for p in SHP_DIR.glob("*.shp"):
    layer_files[p.stem] = p

print("Найденные shapefile:")
for k in sorted(layer_files):
    print(" -", k)

mask_path   = try_find_layer(layer_files, ["svita", "mask"])
facies_path = try_find_layer(layer_files, ["fasi", "fasii", "facies"])
paleo_path  = try_find_layer(layer_files, ["gr_dol", "paleo", "vp_poly"])
struct_path = try_find_layer(layer_files, ["kory", "struct"])
magm_path   = try_find_layer(layer_files, ["dayki", "dike", "magm"])
tect1_path  = try_find_layer(layer_files, ["glub_raz", "tect1", "nw"])
tect2_path  = try_find_layer(layer_files, ["glub_r_", "tect2", "r_nw"])

print("\nАвтоподбор слоев:")
print("mask  :", mask_path.name if mask_path else None)
print("facies:", facies_path.name if facies_path else None)
print("paleo :", paleo_path.name if paleo_path else None)
print("struct:", struct_path.name if struct_path else None)
print("magm  :", magm_path.name if magm_path else None)
print("tect1 :", tect1_path.name if tect1_path else None)
print("tect2 :", tect2_path.name if tect2_path else None)

required = {
    "mask": mask_path,
    "facies": facies_path,
    "paleo": paleo_path,
    "struct": struct_path,
    "magm": magm_path,
    "tect1": tect1_path,
    "tect2": tect2_path,
}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise FileNotFoundError(f"Не найдены обязательные слои: {missing}")


In [ ]:

# =========================
# 6. Загрузка данных
# =========================
layers = {}

layers["mask"] = load_layer(mask_path)
target_crs = layers["mask"].crs

for name, path in required.items():
    if name == "mask":
        continue
    layers[name] = load_layer(path, target_crs=target_crs)

print("CRS mask:", target_crs)
for name, gdf in layers.items():
    print(f"{name:>6}: {len(gdf):>5} объектов | geom types: {sorted(gdf.geom_type.unique().tolist())}")


In [ ]:

# =========================
# 7. Поиск точечных слоев для проверки
# =========================
evidence_candidates = []
for stem, path in layer_files.items():
    if path == mask_path:
        continue
    try:
        tmp = load_layer(path, target_crs=target_crs)
        geom_types = sorted(tmp.geom_type.astype(str).unique().tolist())
        evidence_candidates.append({
            "layer": stem,
            "n": len(tmp),
            "geom_types": geom_types
        })
    except Exception as e:
        evidence_candidates.append({
            "layer": stem,
            "n": -1,
            "geom_types": [f"ERROR: {e}"]
        })

evidence_candidates_df = pd.DataFrame(evidence_candidates).sort_values(["n", "layer"], ascending=[False, True])
display(evidence_candidates_df.head(30))


In [ ]:

# =========================
# 8. Выбор evidence points
# =========================
evidence_paths = []
for stem, path in layer_files.items():
    low = stem.lower()
    if any(k in low for k in ["result", "layer_01", "layer01", "gold", "zolot", "uran", "au", "u_"]):
        evidence_paths.append(path)

evidence_list = []
for p in evidence_paths:
    try:
        g = load_layer(p, target_crs=target_crs)
        g = make_points_from_layer(g)
        if len(g) > 0:
            g["source_layer"] = p.stem
            evidence_list.append(g[["geometry", "source_layer"]].copy())
    except Exception:
        pass

if evidence_list:
    evidence_points = pd.concat(evidence_list, ignore_index=True)
    evidence_points = gpd.GeoDataFrame(evidence_points, geometry="geometry", crs=target_crs)
    evidence_points = evidence_points[evidence_points.geometry.notna()].copy()
    evidence_points.reset_index(drop=True, inplace=True)
else:
    evidence_points = gpd.GeoDataFrame({"source_layer": []}, geometry=[], crs=target_crs)

print("Evidence points:", len(evidence_points))
if len(evidence_points) > 0:
    display(evidence_points.head())


In [ ]:

# =========================
# 9. Построение сетки 500x500
# =========================
grid = build_grid(layers["mask"], CELL_SIZE)
print("Ячеек в сетке:", len(grid))
display(grid.head())


In [ ]:

# =========================
# 10. Расчет расстояний и близостей
# =========================
centroids = gpd.GeoDataFrame(
    grid[["cell_id"]].copy(),
    geometry=grid["centroid"],
    crs=grid.crs
)

unions = {name: union_geometry(gdf) for name, gdf in layers.items() if name != "mask"}

grid["dist_facies"] = distance_to_union(centroids, unions["facies"])
grid["dist_paleo"]  = distance_to_union(centroids, unions["paleo"])
grid["dist_struct"] = distance_to_union(centroids, unions["struct"])
grid["dist_magm"]   = distance_to_union(centroids, unions["magm"])
grid["dist_tect1"]  = distance_to_union(centroids, unions["tect1"])
grid["dist_tect2"]  = distance_to_union(centroids, unions["tect2"])

grid["prox_facies"] = proximity_from_distance(grid["dist_facies"], SCALE_FACIES)
grid["prox_paleo"]  = proximity_from_distance(grid["dist_paleo"],  SCALE_PALEO)
grid["prox_struct"] = proximity_from_distance(grid["dist_struct"], SCALE_STRUCT)
grid["prox_magm"]   = proximity_from_distance(grid["dist_magm"],   SCALE_MAGM)
grid["prox_tect1"]  = proximity_from_distance(grid["dist_tect1"],  SCALE_TECT1)
grid["prox_tect2"]  = proximity_from_distance(grid["dist_tect2"],  SCALE_TECT2)

display(grid[[
    "prox_facies","prox_paleo","prox_struct","prox_magm","prox_tect1","prox_tect2"
]].describe().T)


In [ ]:

# =========================
# 11. Трансформации распределений в духе методички
# =========================
grid["facies_tr"] = np.cbrt(grid["prox_facies"].clip(lower=0))
grid["paleo_tr"]  = np.cbrt(grid["prox_paleo"].clip(lower=0))
grid["tect1_tr"]  = np.cbrt(grid["prox_tect1"].clip(lower=0))
grid["tect2_tr"]  = np.cbrt(grid["prox_tect2"].clip(lower=0))
grid["struct_tr"] = np.sqrt(grid["prox_struct"].clip(lower=0))
grid["magm_tr"]   = np.sqrt(grid["prox_magm"].clip(lower=0))

display(grid[["facies_tr","paleo_tr","tect1_tr","tect2_tr","struct_tr","magm_tr"]].head())


In [ ]:

# =========================
# 12. Производные геологические признаки
# =========================
grid["tect_combo"] = (grid["prox_tect1"] + grid["prox_tect2"]) / 2.0
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = grid["tect_combo"] * grid["prox_magm"]
grid["tect_struct_intersection"] = grid["tect_combo"] * grid["prox_struct"]
grid["paleo_struct_intersection"] = grid["prox_paleo"] * grid["prox_struct"]
grid["paleo_facies_intersection"] = grid["prox_paleo"] * grid["prox_facies"]

factor_cols = ["prox_facies","prox_paleo","prox_struct","prox_magm","prox_tect1","prox_tect2"]
grid["factor_mean"] = grid[factor_cols].mean(axis=1)
grid["factor_min"] = grid[factor_cols].min(axis=1)
grid["factor_std"] = grid[factor_cols].std(axis=1)
grid["factor_balance"] = normalize_01(grid["factor_min"] * (1.0 - grid["factor_std"].fillna(0)))

grid["coincidence_score"] = normalize_01(
    0.25 * grid["tect_intersection"] +
    0.20 * grid["tect_magm_intersection"] +
    0.20 * grid["tect_struct_intersection"] +
    0.15 * grid["paleo_struct_intersection"] +
    0.10 * grid["paleo_facies_intersection"] +
    0.10 * grid["factor_min"]
)

grid["tect_only_penalty"] = normalize_01(
    np.maximum(grid["tect_combo"] - (
        0.35 * grid["prox_magm"] +
        0.35 * grid["prox_struct"] +
        0.15 * grid["prox_paleo"] +
        0.15 * grid["prox_facies"]
    ), 0)
)

grid["support_score"] = normalize_01(
    0.22 * grid["tect_combo"] +
    0.16 * grid["prox_magm"] +
    0.16 * grid["prox_struct"] +
    0.12 * grid["prox_paleo"] +
    0.10 * grid["prox_facies"] +
    0.10 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.06 * grid["tect_struct_intersection"] +
    0.05 * grid["paleo_struct_intersection"] +
    0.05 * grid["factor_balance"] -
    0.10 * grid["tect_only_penalty"]
)

display(grid[[
    "tect_combo","tect_intersection","tect_magm_intersection","tect_struct_intersection",
    "paleo_struct_intersection","paleo_facies_intersection","coincidence_score",
    "factor_mean","factor_min","factor_balance","tect_only_penalty","support_score"
]].head())


In [ ]:

# =========================
# 13. Матрица признаков для unsupervised-методов
# =========================
feature_cols = [
    "facies_tr","paleo_tr","struct_tr","magm_tr","tect1_tr","tect2_tr",
    "tect_combo","tect_intersection","tect_magm_intersection","tect_struct_intersection",
    "paleo_struct_intersection","paleo_facies_intersection",
    "coincidence_score","factor_mean","factor_min","factor_balance","tect_only_penalty"
]

X_raw = grid[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
scaler = StandardScaler()
X_model = scaler.fit_transform(X_raw)

print("Форма матрицы признаков:", X_model.shape)
print("Используемые признаки:")
for c in feature_cols:
    print(" -", c)


In [ ]:

# =========================
# 14. PCA-диагностика
# =========================
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_model)

grid["pca1"] = X_pca[:, 0]
grid["pca2"] = X_pca[:, 1]

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(
    grid["pca1"], grid["pca2"],
    c=grid["support_score"],
    s=9,
    alpha=0.8
)
ax.set_title("PCA: структура ячеек по геологическим признакам")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("support_score")
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)


In [ ]:

# =========================
# 15. Autoencoder (MLPRegressor)
# =========================
ae = MLPRegressor(
    hidden_layer_sizes=AE_HIDDEN,
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=AE_MAX_ITER,
    random_state=RANDOM_STATE
)

ae.fit(X_model, X_model)
X_recon = ae.predict(X_model)

recon_error = np.mean((X_model - X_recon) ** 2, axis=1)
grid["autoencoder_score"] = normalize_01(recon_error)

print("Autoencoder обучен.")
print("Средний reconstruction error:", float(np.mean(recon_error)))
print("Min/Max error:", float(np.min(recon_error)), float(np.max(recon_error)))


In [ ]:

# =========================
# 16. GMM с выбором числа компонент по BIC
# =========================
best_gmm = None
best_bic = np.inf
best_k = None

for k in GMM_COMPONENTS_RANGE:
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=RANDOM_STATE,
        reg_covar=1e-6
    )
    gmm.fit(X_model)
    bic = gmm.bic(X_model)
    print(f"k={k}: BIC={bic:.2f}")
    if bic < best_bic:
        best_bic = bic
        best_gmm = gmm
        best_k = k

print("\nЛучшая модель GMM:")
print("n_components =", best_k)
print("best BIC     =", best_bic)


In [ ]:

# =========================
# 17. Вероятностный GMM-score
# =========================
gmm_labels = best_gmm.predict(X_model)
gmm_proba = best_gmm.predict_proba(X_model)

grid["gmm_cluster"] = gmm_labels

cluster_info = pd.DataFrame({
    "cluster": gmm_labels,
    "support_score": grid["support_score"].values,
    "factor_balance": grid["factor_balance"].values,
    "coincidence_score": grid["coincidence_score"].values
})

cluster_quality_series = (
    0.45 * cluster_info.groupby("cluster")["support_score"].mean() +
    0.30 * cluster_info.groupby("cluster")["coincidence_score"].mean() +
    0.25 * cluster_info.groupby("cluster")["factor_balance"].mean()
).sort_index()

cluster_quality = normalize_01(cluster_quality_series.values)
grid["gmm_score"] = normalize_01(gmm_proba @ cluster_quality)

display(pd.DataFrame({
    "cluster": cluster_quality_series.index,
    "cluster_quality_raw": cluster_quality_series.values,
    "cluster_quality_norm": cluster_quality,
    "count": pd.Series(gmm_labels).value_counts().sort_index().values
}))


In [ ]:

# =========================
# 18. Isolation Forest
# =========================
iforest = IsolationForest(
    n_estimators=400,
    contamination=IFOREST_CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iforest.fit(X_model)

iso_raw = -iforest.score_samples(X_model)
grid["isolation_score"] = normalize_01(iso_raw)

print("Isolation Forest готов.")


In [ ]:

# =========================
# 19. Итоговый unsupervised ensemble
# =========================
unsup_core = (
    0.35 * grid["autoencoder_score"] +
    0.25 * grid["gmm_score"] +
    0.25 * grid["isolation_score"] +
    0.15 * grid["support_score"]
)

support_gate = np.power(grid["support_score"].clip(0, 1), SUPPORT_POWER)

grid["unsup_core"] = normalize_01(unsup_core)
grid["prospectivity"] = normalize_01(grid["unsup_core"] * support_gate)

grid["prognoz"] = 1.0 - grid["prospectivity"]

top_thr = grid["prospectivity"].quantile(TOP_Q)
grid["gold_zone"] = (
    (grid["prognoz"] <= YELLOW_LIMIT) &
    (grid["prospectivity"] >= top_thr)
).astype(int)

print("TOP_Q threshold:", float(top_thr))
print("Gold-zone cells:", int(grid["gold_zone"].sum()))
print("Gold-zone share:", float(grid["gold_zone"].mean()))


In [ ]:

# =========================
# 20. Проверка по известным точкам
# =========================
metrics = {}

if len(evidence_points) > 0:
    joined = gpd.sjoin(
        evidence_points,
        grid[["cell_id", "prospectivity", "prognoz", "gold_zone", "geometry"]],
        how="left",
        predicate="within"
    )

    metrics["evidence_points_total"] = int(len(evidence_points))
    metrics["evidence_points_joined"] = int(joined["cell_id"].notna().sum())
    metrics["gold_zone_hits"] = int((joined["gold_zone"] == 1).sum())

    for q in [0.90, 0.85, 0.80]:
        thr = grid["prospectivity"].quantile(q)
        hit = int((joined["prospectivity"] >= thr).sum())
        metrics[f"HitRate_top_{int((1-q)*100):02d}pct"] = float(hit / max(len(joined), 1))
        metrics[f"Hits_top_{int((1-q)*100):02d}pct"] = hit

    print("Метрики попадания по известным точкам:")
    for k, v in metrics.items():
        print(f"{k}: {v}")
else:
    print("Evidence points не найдены — расчёт hit-rate пропущен.")


In [ ]:

# =========================
# 21. Гистограммы оценок
# =========================
plt.figure(figsize=(12, 7))
for col in ["autoencoder_score", "gmm_score", "isolation_score", "support_score", "prospectivity"]:
    plt.hist(grid[col], bins=35, alpha=0.45, label=col)

plt.title("Распределения оценок методов")
plt.xlabel("score")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()

hist_png = OUT_DIR / "score_distributions.png"
plt.savefig(hist_png, dpi=250, bbox_inches="tight")
plt.show()

print("Сохранено:", hist_png)


In [ ]:

# =========================
# 22. Финальная карта в стиле примера
# =========================
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np

# Защита от проблем с aspect в GeoPandas/Matplotlib
plot_grid = grid.copy()
plot_grid = plot_grid[np.isfinite(plot_grid["prognoz"])].copy()
plot_grid = plot_grid[plot_grid.geometry.notnull()].copy()
plot_grid = plot_grid[~plot_grid.geometry.is_empty].copy()

bounds = [
    0.00, 0.15,
    0.25, 0.35, 0.45,
    0.55, 0.65, 0.75,
    0.85, 1.00
]

colors = [
    "#ffe600",
    "#2736a6",
    "#4857c7",
    "#7f8fe0",
    "#dce2ff",
    "#ffffff",
    "#ffd6d6",
    "#ff9a9a",
    "#ef3b3b"
]

cmap = ListedColormap(colors)
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(10, 12))
ax.set_aspect("auto")

plot_grid.plot(
    column="prognoz",
    cmap=cmap,
    norm=norm,
    linewidth=0,
    ax=ax,
    aspect="auto",
    legend=True,
    legend_kwds={
        "label": "prognoz: 0 — high prospectivity, 1 — low",
        "shrink": 0.72
    }
)

for name in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    try:
        layer_plot = layers[name].copy()
        layer_plot = layer_plot[layer_plot.geometry.notnull()]
        layer_plot = layer_plot[~layer_plot.geometry.is_empty]
        if len(layer_plot) > 0:
            layer_plot.boundary.plot(
                ax=ax,
                color="black",
                linewidth=0.45,
                alpha=0.55,
                aspect="auto"
            )
    except Exception:
        pass

try:
    ep = evidence_points.copy()
    ep = ep[ep.geometry.notnull()]
    ep = ep[~ep.geometry.is_empty]
    if len(ep) > 0:
        ep.plot(
            ax=ax,
            color="yellow",
            edgecolor="black",
            linewidth=0.35,
            markersize=14,
            alpha=0.95,
            aspect="auto"
        )
except Exception:
    pass

ax.set_title(
    "Unsupervised forecast: Autoencoder + GMM + Isolation Forest",
    fontsize=14
)
ax.set_axis_off()
plt.tight_layout()

map_png = OUT_DIR / "unsupervised_forecast_gis_style.png"
plt.savefig(map_png, dpi=300, bbox_inches="tight")
plt.show()

print("Карта сохранена:", map_png)
print("Нарисовано ячеек:", len(plot_grid))


In [ ]:

# =========================
# 23. Отдельный слой жёлтых зон
# =========================
gold_cells = grid[grid["gold_zone"] == 1].copy()
print("Жёлтых ячеек:", len(gold_cells))

fig, ax = plt.subplots(figsize=(8, 10))
grid.boundary.plot(ax=ax, color="lightgray", linewidth=0.2)
if len(gold_cells) > 0:
    gold_cells.plot(ax=ax, color="#ffe600", edgecolor="black", linewidth=0.15)
if len(evidence_points) > 0:
    evidence_points.plot(ax=ax, color="red", markersize=8, alpha=0.8)
ax.set_title("Top unsupervised zones")
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:

# =========================
# 24. Сохранение результатов
# =========================
grid_out = grid.drop(columns=["centroid"]).copy()

gpkg_path = OUT_DIR / "unsupervised_forecast_result.gpkg"
csv_path = OUT_DIR / "grid_attributes.csv"
json_path = OUT_DIR / "metrics.json"

grid_out.to_file(gpkg_path, driver="GPKG", layer="grid_result")
if len(evidence_points) > 0:
    evidence_points.to_file(gpkg_path, driver="GPKG", layer="evidence_points")
if len(gold_cells) > 0:
    gold_cells.to_file(gpkg_path, driver="GPKG", layer="top_zones")

grid_out.drop(columns="geometry").to_csv(csv_path, index=False, encoding="utf-8-sig")

summary = {
    "n_cells": int(len(grid_out)),
    "n_gold_cells": int(grid_out["gold_zone"].sum()),
    "gold_zone_share": float(grid_out["gold_zone"].mean()),
    "TOP_Q": float(TOP_Q),
    "YELLOW_LIMIT": float(YELLOW_LIMIT),
    "AE_HIDDEN": list(AE_HIDDEN),
    "GMM_best_components": int(best_k),
    "IFOREST_CONTAMINATION": float(IFOREST_CONTAMINATION),
    "feature_cols": feature_cols,
    "metrics": metrics
}
save_json(summary, json_path)

print("Сохранено:")
print(" -", gpkg_path)
print(" -", csv_path)
print(" -", json_path)


In [ ]:

# =========================
# 25. Краткая диагностика
# =========================
diag_cols = [
    "prospectivity", "prognoz", "autoencoder_score", "gmm_score",
    "isolation_score", "support_score", "gold_zone"
]
display(grid[diag_cols].describe().T)

print("\nТОП-20 ячеек по prospectivity:")
display(
    grid[["cell_id", "prospectivity", "prognoz", "support_score", "autoencoder_score", "gmm_score", "isolation_score"]]
    .sort_values("prospectivity", ascending=False)
    .head(20)
)


In [ ]:

# =========================
# 26. Что можно подкрутить
# =========================
# 1) Если жёлтых зон слишком много:
#    - увеличь TOP_Q до 0.95 или 0.96
#    - или уменьшай YELLOW_LIMIT до 0.12
#
# 2) Если карта слишком "синяя":
#    - уменьши SUPPORT_POWER до 0.55
#
# 3) Если карта слишком "шумная":
#    - слегка увеличь SCALE_* или попробуй более мягкое объединение в support_score
#
# 4) Если GMM даёт слишком резкие области:
#    - попробуй диапазон GMM_COMPONENTS_RANGE = range(2, 6)
#
# 5) Главное:
#    - не добавляй pseudo-labels
#    - не используй RandomForest / LogisticRegression в этой версии
#
print("Ноутбук выполнен.")
